In [47]:
# Case Study — Group 11

## 1. Introduction
### 1.1 Objective
### 1.2 Interpretation of the Task
### 1.3 Analysis and Data Selection Strategy

## 2. Project Setup
### 2.1 Imports and Configuration
### 2.2 File Structure and Paths

## 3. Importing and Exploring the Data
### 3.1 Vehicle Data
### 3.2 Vehicle Component Lists
### 3.3 Engine Data
### 3.4 Gearshift Data
### 3.5 Component Part Lists
### 3.6 Single-Part Data
### 3.7 Geodata and Registration Data

## 4. Data Preparation
### 4.1 Selection of Vehicles Produced in 2015
### 4.2 Selection of Engine and Gearshift Components
### 4.3 Selection of Corresponding Parts
### 4.4 Data Cleaning and Validation

## 5. Data Integration and Supply Chain Reconstruction
### 5.1 Vehicles and Components
### 5.2 Components and Parts
### 5.3 Production Locations
### 5.4 Distribution Centers and Customers
### 5.5 Merge Validation

# Case Study — Group 11

The task is to reconstruct the supply chain of vehicles produced by OEM1 in 2015 and to determine the total logistics distance travelled before the vehicles reach their customers. The analysis focuses on the engine and gearshift components as well as the corresponding parts installed in these components. For each relevant vehicle, the routes from the parts suppliers to the component suppliers, from the component suppliers to the OEM1 production plant, and from the production plant via the responsible distribution center to the customer are identified.

## Setup

The original folders are stored below `data`. Needed modules are imported and needed functions are defined.

In [28]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
RANDOM_STATE = 42

def read_matches(path, usecols, column, values, sep=";", chunksize=250_000):
    """Read only rows whose key occurs in values, without changing the file."""
    matches = []
    for chunk in pd.read_csv(
        path,
        sep=sep,
        usecols=usecols,
        dtype="string",
        chunksize=chunksize,
    ):
        selected = chunk.loc[chunk[column].isin(values)].copy()
        if not selected.empty:
            matches.append(selected)
    return (
        pd.concat(matches, ignore_index=True)
        if matches
        else pd.DataFrame(columns=usecols)
    )

Since only OEM1 vehicles produced in 2015 are considered, the first step is to import the OEM1 (Typ11 and Typ12) vehicle datasets and filter them by production year. The X1 column can be omitted because it duplicates the index column. Furthermore, information about whether a vehicle is marked as Fehlerhaft (defective) is not relevant for this analysis. Therefore, the corresponding defect-related columns are excluded as well.

In [26]:
chunks = pd.read_csv(
    "data/IDA SoSe26 - Data/Fahrzeug/Fahrzeuge_OEM1_Typ11.csv",
    usecols=["ID_Fahrzeug", "Produktionsdatum", "Herstellernummer", "Werksnummer"],
    parse_dates=["Produktionsdatum"],
    chunksize=100_000
)

OEM1_Typ11 = pd.concat(
    chunk[chunk["Produktionsdatum"].dt.year == 2015]
    for chunk in chunks
)

OEM1_Typ11.info()

<class 'pandas.DataFrame'>
RangeIndex: 247868 entries, 1509823 to 1757690
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   ID_Fahrzeug       247868 non-null  str           
 1   Produktionsdatum  247868 non-null  datetime64[us]
 2   Herstellernummer  247868 non-null  int64         
 3   Werksnummer       247868 non-null  int64         
dtypes: datetime64[us](1), int64(2), str(1)
memory usage: 10.9 MB


In [27]:
chunks = pd.read_csv(
    "data/IDA SoSe26 - Data/Fahrzeug/Fahrzeuge_OEM1_Typ12.csv",
    usecols=["ID_Fahrzeug", "Produktionsdatum", "Herstellernummer", "Werksnummer"],
    sep = ";",
    parse_dates=["Produktionsdatum"],
    chunksize=100_000
)

OEM1_Typ12 = pd.concat(
    chunk[chunk["Produktionsdatum"].dt.year == 2015]
    for chunk in chunks
)

OEM1_Typ12.info()

<class 'pandas.DataFrame'>
RangeIndex: 51385 entries, 311124 to 362508
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ID_Fahrzeug       51385 non-null  str           
 1   Produktionsdatum  51385 non-null  datetime64[us]
 2   Herstellernummer  51385 non-null  int64         
 3   Werksnummer       51385 non-null  int64         
dtypes: datetime64[us](1), int64(2), str(1)
memory usage: 2.3 MB


Each `OEM1_Typ11` and `OEM1_Typ12` dataset is merged with its corresponding `Bestandteile_Fahrzeug` relation table to identify the relevant gearshift and engine components installed in each vehicle. Since only these two component types are relevant for the analysis, all other components are excluded. The resulting datasets are then concatenated, while adding a `Type` column to preserve the distinction between vehicle types.

In [33]:
typ11_components = read_matches(
    path="data/IDA SoSe26 - Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv",
    usecols=["ID_Fahrzeug", "ID_Motor", "ID_Schaltung"],
    column="ID_Fahrzeug",
    values=set(OEM1_Typ11["ID_Fahrzeug"])
)

In [39]:
component_merge_typ11 = OEM1_Typ11.merge(typ11_components, how = "left", on = "ID_Fahrzeug", validate = "1:1")

In [38]:
component_merge_typ11.isna().sum()

ID_Fahrzeug         0
Produktionsdatum    0
Herstellernummer    0
Werksnummer         0
ID_Schaltung        0
ID_Motor            0
dtype: int64

In [42]:
typ12_components = read_matches(
    path="data/IDA SoSe26 - Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv",
    usecols=["ID_Fahrzeug", "ID_Motor", "ID_Schaltung"],
    column="ID_Fahrzeug",
    values=set(OEM1_Typ12["ID_Fahrzeug"])
)

In [44]:
component_merge_typ12 = OEM1_Typ12.merge(typ12_components, how = "left", on = "ID_Fahrzeug", validate = "1:1")

In [46]:
component_merge_typ12.isna().sum()

ID_Fahrzeug         0
Produktionsdatum    0
Herstellernummer    0
Werksnummer         0
ID_Schaltung        0
ID_Motor            0
dtype: int64